# Pré-processamento das Bases da SSP-SP
O objetivo é consolidar as fontes obtidas pela transparência da **Secretária de Segurança Pública de São Paulo (SPP-SP)** em um único **dataset**.

## 1. Bibliotecas e Configuração Inicial

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
np.random.seed(42)

### 1.1 Fontes de Dados

São listados os arquivos brutos (`.xlsx`) disponibilizados pela transparência da SSP-SP, um por ano, de **2022 a 2026**. Cada arquivo contém os Boletins de Ocorrência (B.O.) registrados no respectivo período.

In [3]:
data_path = '../datasets/'
ssp_data_path = data_path + 'SSP-SP/'

In [ ]:
files = [ssp_data_path + 'SPDadosCriminais_2022.xlsx',
         ssp_data_path + 'SPDadosCriminais_2023.xlsx',
         ssp_data_path + 'SPDadosCriminais_2024.xlsx',
         ssp_data_path + 'SPDadosCriminais_2025.xlsx',
         ssp_data_path + 'SPDadosCriminais_2026.xlsx']

In [ ]:
list_dataframes = []

### 1.2 Leitura e Consolidação das Planilhas

Cada arquivo Excel possui **múltiplas abas** (uma por semestre). O código:

1. Lê **todas as abas** de cada arquivo (`sheet_name=None`), retornando um dicionário `{nome_da_aba: dataframe}`;
2. **Ignora a primeira aba** (`abas[1:]`), que corresponde a uma folha de rosto/legenda, e não a dados propriamente ditos;
3. Adiciona duas colunas de rastreabilidade em cada dataframe — `origin_file` e `origin_aba` — para manter o registro de **qual arquivo e aba deram origem a cada linha**;
4. Acumula todos os dataframes resultantes em `list_dataframes`.

In [ ]:
for file in files:
    abas_dict = pd.read_excel(file, sheet_name=None)
    abas = list(abas_dict.items())
    for name_aba, df_aba in abas[1:]:
        df_aba['origin_file'] = file
        df_aba['origin_aba'] = name_aba
        list_dataframes.append(df_aba)

### 1.3 Concatenação da Base Bruta

Todos os dataframes (de todos os anos e abas) são concatenados verticalmente em um único dataframe (`df_raw`), consolidando os B.O. de 2022 a 2026 em uma única estrutura de dados.

In [ ]:
df_raw = pd.concat(list_dataframes, ignore_index=True)

In [ ]:
df_raw.head()

### 1.4 Salvar a base

In [ ]:
df_raw.to_csv(ssp_data_path + 'SPDadosCriminais_2022_2026.csv', index=False)

### 1.5 Leitura da base

É necessário gerar a quantidade de ocorrências por tipo de crime, localização, quinzena, mês e ano para subsidiar o cálculo do risco de <b>criminalidade</b>.

In [4]:
df_raw = pd.read_csv(ssp_data_path + 'SPDadosCriminais_2022_2026.csv')

C:\Users\Usuario\AppData\Local\Temp\ipykernel_19588\4253070456.py:1: DtypeWarning: Columns (3,6,9,13,14,15,16,17,18,19,31,32,33,34,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(ssp_data_path + 'SPDadosCriminais_2022_2026.csv')


In [5]:
print(f"Tamanho da base original = {len(df_raw):}".replace(",", ".") + " registros")

Tamanho da base original = 5348234 registros


## 2. Base de dados socioeconômicos

Utilizam-se dados do IPVS e do IBGE, relacionados ao Censo de 2022, para incorporar características estruturais do território e avaliar se elas melhoram o desempenho na predição do risco de roubo e furto.

**Base do IPVS 2022**

In [6]:
gdf_ipvs = gpd.read_file(data_path + 'IPVS_2022/IPVS_2022.shp')

In [7]:
gdf_ipvs.head()

,fid,CD_SETOR,SITUACAO,CD_MUN,NM_MUN,CD_DIST,NM_DIST,N_IPVS,C_IPVS,geometry
0,1.0,355030811000549,Urbana,3550308,São Paulo,355030811,Brasilândia,Média Vulnerabilidade,4.0,"POLYGON ((-46.69624 -23.456, -46.69629 -23.456..."
1,2.0,355030811000548,Urbana,3550308,São Paulo,355030811,Brasilândia,Média Vulnerabilidade,4.0,"POLYGON ((-46.69781 -23.45723, -46.69775 -23.4..."
2,3.0,354160405000053,Urbana,3541604,Promissão,354160405,Promissão,Não classificado,NaN,"POLYGON ((-49.7252 -21.40299, -49.72527 -21.40..."
3,4.0,354160405000052,Urbana,3541604,Promissão,354160405,Promissão,Não classificado,NaN,"POLYGON ((-49.72019 -21.4131, -49.72079 -21.41..."
4,5.0,353440105000527,Urbana,3534401,Osasco,353440105,Osasco,Muito Baixa Vulnerabilidade,2.0,"POLYGON ((-46.81583 -23.53075, -46.8155 -23.53..."


In [8]:
print("Colunas do IPVS:", gdf_ipvs.columns.tolist())

Colunas do IPVS: ['fid', 'CD_SETOR', 'SITUACAO', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'N_IPVS', 'C_IPVS', 'geometry']


In [9]:
def read_census_sp(file_csv, maps_columns):
    header = pd.read_csv(file_csv, sep=';', nrows=0, encoding='latin1').columns.tolist()
    col_map_real = {c.upper().strip(): c for c in header}
    
    if 'CD_SETOR' in col_map_real:
        chave_setor = 'CD_SETOR'
    elif 'SETOR' in col_map_real:
        chave_setor = 'SETOR'
    else:
        raise KeyError(f"Nenhuma coluna de setor ('CD_SETOR' ou 'SETOR') encontrada em {file_csv}")

    cols = [chave_setor] + list(maps_columns.keys())
    cols_reais = [col_map_real[c.upper()] for c in cols if c.upper() in col_map_real]
    
    df = pd.read_csv(file_csv, sep=';', usecols=cols_reais, dtype=str, encoding='latin1')
    df.columns = [c.upper().strip() for c in df.columns]
    
    if chave_setor == 'SETOR':
        df = df.rename(columns={'SETOR': 'CD_SETOR'})
        
    df['CD_SETOR'] = df['CD_SETOR'].str.strip()

    STATE_SP = '35'
    df = df[df['CD_SETOR'].str.startswith(STATE_SP, na=False)].copy()
    
    for col in maps_columns.keys():
        col_up = col.upper()
        if col_up in df.columns:
            df[col_up] = pd.to_numeric(df[col_up].str.replace(',', '.'), errors='coerce')
            
    return df.rename(columns={k.upper(): v for k, v in maps_columns.items()})

**Base de Agregados por Setores Censitários de 2022**

In [10]:
df_basico = read_census_sp(data_path + 'CENSO_IBGE_2022/Agregados_por_setores_basico_BR.csv', {
    'V0001': 'pop_total', 'V0002': 'domicilios_total', 'V0005': 'media_moradores', 'V0009': 'domicilios_vagos'
})

In [11]:
df_demo = read_census_sp(data_path + 'CENSO_IBGE_2022/Agregados_por_setores_demografia_BR.csv', {
    'V01007': 'pop_masc', 'V01008': 'pop_fem', 'V01034': 'jovens_15_19', 'V01035': 'jovens_20_24', 'V01036': 'jovens_25_29'
})

In [12]:
df_alfab = read_census_sp(data_path + 'CENSO_IBGE_2022/Agregados_por_setores_alfabetizacao_BR.csv', {
    'V00900': 'alfabetizados_15mais', 'V00901': 'nao_alfabetizados_15mais'
})

In [13]:
df_raca = read_census_sp(data_path + 'CENSO_IBGE_2022/Agregados_por_setores_cor_ou_raca_BR.csv', {
    'V01317': 'pop_branca', 'V01318': 'pop_preta', 'V01320': 'pop_parda'
})

In [14]:
df_dom1 = read_census_sp(data_path + 'CENSO_IBGE_2022/Agregados_por_setores_caracteristicas_domicilio1_BR_20261021.csv', {
    'V00047': 'dom_casa', 'V00049': 'dom_apartamento', 'V00050': 'dom_cortico', 'V00052': 'dom_degradado'
})

In [15]:
df_dom2 = read_census_sp(data_path + 'CENSO_IBGE_2022/Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv', {
    'V00199': 'dom_agua_encanada', 'V00309': 'dom_esgoto_rede', 'V00397': 'dom_lixo_coletado'
})

In [16]:
gdf_setores_completo = gdf_ipvs.copy()

for df_tema in [df_basico, df_demo, df_alfab, df_raca, df_dom1, df_dom2]:
    gdf_setores_completo = gdf_setores_completo.merge(df_tema, on='CD_SETOR', how='left')

Cálculo de variáveis percentuais

In [17]:
gdf_setores_completo['pct_jovens_15_29'] = (
    (gdf_setores_completo['jovens_15_19'] + gdf_setores_completo['jovens_20_24'] + gdf_setores_completo['jovens_25_29']) 
    / gdf_setores_completo['pop_total'] * 100
)

In [18]:
gdf_setores_completo['taxa_alfabetizacao'] = (
    gdf_setores_completo['alfabetizados_15mais'] 
    / (gdf_setores_completo['alfabetizados_15mais'] + gdf_setores_completo['nao_alfabetizados_15mais']) * 100
)

In [19]:
gdf_setores_completo['pct_negros'] = (
    (gdf_setores_completo['pop_preta'] + gdf_setores_completo['pop_parda']) 
    / gdf_setores_completo['pop_total'] * 100
)

Não há setores censitários duplicados (`CD_SETOR`)

In [20]:
cnt_dedup_records_setores = gdf_setores_completo.duplicated(subset=['CD_SETOR']).sum()
pct_dedup_records_setores = cnt_dedup_records_setores / len(gdf_setores_completo) * 100
print(f"Frequência de Registros Duplicados = {str(pct_dedup_records_setores).replace('.', ',')}% " + f"({cnt_dedup_records_setores:})".replace(",", "."))

Frequência de Registros Duplicados = 0,0% (0)


## 3. Padronização da Coluna de Cidade

A base possui duas colunas que podem indicar o município do B.O.: `NOME_MUNICIPIO_CIRCUNCRIÇÃO` (município da circunscrição/delegacia responsável [local onde ocorreu o fato]) e `NOME_MUNICIPIO` (município registrado). Como a primeira apresenta uma quantidade relevante de nulos, é necessário avaliar se a segunda pode servir de **fallback**, e então consolidar as duas em uma única coluna `CIDADE`.

In [21]:
len(df_raw[df_raw['NOME_MUNICIPIO_CIRCUNCRIÇÃO'].isnull()]) / len(df_raw)

0.10384811135788

In [22]:
city_null = df_raw[df_raw['NOME_MUNICIPIO_CIRCUNCRIÇÃO'].isnull()]

In [23]:
len(city_null[city_null['NOME_MUNICIPIO'].isnull()]) / len(city_null)

0.0

Confirmada a viabilidade do fallback, a coluna `CIDADE` é criada a partir de `NOME_MUNICIPIO_CIRCUNCRIÇÃO`, preenchendo os valores nulos com `NOME_MUNICIPIO`.

In [24]:
df_raw['CIDADE'] = df_raw['NOME_MUNICIPIO_CIRCUNCRIÇÃO'].fillna(df_raw['NOME_MUNICIPIO'])

In [25]:
df_raw['CIDADE'] = df_raw['CIDADE'].str.strip()

Coluna `CIDADE` sem a presença de valores nulos.

In [26]:
len(df_raw[df_raw['CIDADE'].isnull()]) / len(df_raw)

0.0

## 4. Filtragem por Tipo de Crime

O estudo tem como foco **Roubos e Furtos**. Antes de aplicar qualquer filtro, é feito um levantamento dos valores únicos da coluna `NATUREZA_APURADA`, para identificar todas as categorias existentes e decidir quais devem ser mantidas.

In [27]:
df_raw['NATUREZA_APURADA'].unique()

array(['ESTUPRO', 'FURTO - OUTROS', 'FURTO DE VEÍCULO',
       'HOMICÍDIO DOLOSO',
       'LESÃO CORPORAL CULPOSA POR ACIDENTE DE TRÂNSITO',
       'LESÃO CORPORAL DOLOSA', 'ROUBO - OUTROS', 'ROUBO DE CARGA',
       'ROUBO DE VEÍCULO', 'TENTATIVA DE HOMICIDIO',
       'TRAFICO DE ENTORPECENTES', 'LESÃO CORPORAL CULPOSA – OUTRAS',
       'ESTUPRO DE VULNERÁVEL', 'FURTO DE CARGA',
       'HOMICIDIO CULPOSO POR ACIDENTE DE TRANSITO',
       'HOMICIDIO CULPOSO OUTROS', 'EXTORSÃO MEDIANTE SEQUESTRO',
       'LATROCÍNIO', 'ROUBO A BANCO', 'LESÃO CORPORAL SEGUIDA DE MORTE',
       'HOMICÍDIO DOLOSO POR ACIDENTE DE TRÂNSITO',
       'HOMICÍDIO CULPOSO POR ACIDENTE DE TRÂNSITO',
       'LESÃO CORPORAL CULPOSA - OUTRAS', 'PORTE DE ARMA',
       'PORTE DE ENTORPECENTES', 'TRÁFICO DE ENTORPECENTES',
       'APREENSÃO DE ENTORPECENTES', 'TENTATIVA DE HOMICÍDIO',
       'HOMICÍDIO CULPOSO OUTROS', 'Furto de Veículo', 'Roubo de Veículo',
       'Furto - Outros', 'Estupro', 'Estupro de Vulnerável',
  

Como o objetivo do estudo é analisar **Roubos e Furtos**, serão mantidas as seguintes categorias:
- 'FURTO - OUTROS'
- 'FURTO DE VEÍCULO'
- 'FURTO DE CARGA'
- 'ROUBO - OUTROS'
- 'ROUBO DE CARGA'
- 'ROUBO DE VEÍCULO'
- 'ROUBO A BANCO'
<p>Em seguida, <b>'ROUBO A BANCO'</b> será incorporado em <b>'ROUBO - OUTROS'</b></p>

In [28]:
list_crimes = ['FURTO - OUTROS', 'FURTO DE VEÍCULO', 'ROUBO - OUTROS',
               'ROUBO DE CARGA', 'ROUBO DE VEÍCULO', 'FURTO DE CARGA',
                'ROUBO A BANCO', 'Furto de Veículo', 'Roubo de Veículo',
                    'Furto - Outros']

In [29]:
df_raw = df_raw[df_raw['NATUREZA_APURADA'].isin(list_crimes)]

In [30]:
df_raw['NATUREZA_APURADA'] = df_raw['NATUREZA_APURADA'].str.upper()

In [31]:
df_raw['NATUREZA_APURADA'] = df_raw['NATUREZA_APURADA'].map(lambda x: 'ROUBO - OUTROS' if x == 'ROUBO A BANCO' else x)

In [32]:
df_raw['NATUREZA_APURADA'].unique()

array(['FURTO - OUTROS', 'FURTO DE VEÍCULO', 'ROUBO - OUTROS',
       'ROUBO DE CARGA', 'ROUBO DE VEÍCULO', 'FURTO DE CARGA'],
      dtype=object)

Será mantido apenas B.O. de munícipios de Região Metropolitana de São Paulo. A lista de munícipios está disponível em: https://pt.wikipedia.org/wiki/Regi%C3%A3o_Metropolitana_de_S%C3%A3o_Paulo

## 5. Filtragem Geográfica: Região Metropolitana de São Paulo

O escopo do estudo é restrito aos municípios da **Região Metropolitana de São Paulo (RMSP)**. A lista de municípios é definida manualmente com base na composição oficial da RMSP, e utilizada para filtrar a base pela coluna `CIDADE`.

In [33]:
list_city = ['ARUJA', 'BARUERI', 'BIRITIBA-MIRIM', 'CAIEIRAS', 'CAJAMAR',
             'CARAPICUIBA', 'COTIA', 'DIADEMA', 'EMBU DAS ARTES', 'EMBU-GUACU',
                'FERRAZ DE VASCONCELOS', 'FRANCISCO MORATO', 'FRANCO DA ROCHA', 'GUARAREMA',
                    'GUARULHOS', 'ITAPECERICA DA SERRA', 'ITAPEVI', 'ITAQUAQUECETUBA',
                        'JANDIRA', 'JUQUITIBA', 'MAIRIPORA', 'MAUA', 'MOGI DAS CRUZES', 'OSASCO',
                        'PIRAPORA BOM JESUS', 'POA', 'RIBEIRAO PIRES', 'RIO GRANDE DA SERRA',
                            'SALESOPOLIS', 'SANTANA DE PARNAIBA', 'S.ANDRE', 'S.BERNARDO DO CAMPO', 'S.CAETANO DO SUL', 'S.LOURENCO DA SERRA', 'S.PAULO', 'SUZANO', 'TABOAO DA SERRA', 'VARGEM GRANDE PAULISTA']

In [34]:
df_raw = df_raw[df_raw['CIDADE'].isin(list_city)]

## 6. Tratamento das Datas de Ocorrência

Antes de derivar mês, dia e ano do B.O., é avaliada a completude da coluna `DATA_OCORRENCIA_BO` e `DATA_REGISTRO`.

Frequência de valores nulos em `DATA_OCORRENCIA_BO`

In [35]:
pct_data_ocorrencia_bo = len(df_raw[df_raw['DATA_OCORRENCIA_BO'].isnull()]) / len(df_raw) * 100
print(f"{str(pct_data_ocorrencia_bo).replace('.', ',')}% " + f"({len(df_raw[df_raw['DATA_OCORRENCIA_BO'].isnull()]):})".replace(",", "."))

0,016727296723061186% (436)


Frequência de valores nulos em `DATA_REGISTRO`

In [36]:
pct_data_registro = len(df_raw[df_raw['DATA_REGISTRO'].isnull()]) / len(df_raw) * 100
print(f"{str(pct_data_registro).replace('.', ',')}% " + f"({len(df_raw[df_raw['DATA_REGISTRO'].isnull()]):})".replace(",", "."))

23,576856173638546% (614535)


Devido ao baixo volume, registros sem `DATA_OCORRENCIA_BO` serão excluídos.

In [37]:
df_raw = df_raw[df_raw['DATA_OCORRENCIA_BO'].notnull()]

`DATA_OCORRENCIA_BO` é convertids para o tipo `datetime`, com `errors='coerce'` transformando qualquer valor inválido em nulo (`NaT`), em vez de interromper a execução.

In [38]:
df_raw['DATA_OCORRENCIA_BO'] = pd.to_datetime(df_raw['DATA_OCORRENCIA_BO'], errors='coerce')

### 6.1 Derivando Ano, Mês e Dia

As colunas `ANO_BO`, `MES_BO` e `DIA_BO` são extraídas de `DATA_OCORRENCIA_BO`.

In [39]:
df_raw['ANO_BO'] = df_raw['DATA_OCORRENCIA_BO'].dt.year.astype('Int64')
df_raw['MES_BO'] = df_raw['DATA_OCORRENCIA_BO'].dt.month.astype('Int64')
df_raw['DIA_BO'] = df_raw['DATA_OCORRENCIA_BO'].dt.day.astype('Int64')

**Filtro aplicado:** registros com `ANO_BO <= 2021` foram descartados como ruído, assim como registros com `ANO_BO = 2026` e `MES_BO = 7`. Sua incidência é baixa, inferior à taxa de dados ausentes em `DATA_REGISTRO`.

In [40]:
pct_year_bo_less_2021 = len(df_raw[df_raw['ANO_BO'] <= 2021]) / len(df_raw) * 100
print(f"{str(pct_year_bo_less_2021).replace('.', ',')}% " + f"({len(df_raw[df_raw['ANO_BO'] <= 2021]):})".replace(",", "."))

0,5639116497485498% (14696)


In [41]:
pct_year_2026_mounth_7 = len(df_raw[((df_raw['ANO_BO'] == 2026) & (df_raw['MES_BO'] == 7))]) / len(df_raw) * 100
print(f"{str(pct_year_2026_mounth_7).replace('.', ',')}% " + f"({len(df_raw[((df_raw['ANO_BO'] == 2026) & (df_raw['MES_BO'] == 7))]):})".replace(",", "."))

0,00011511533405318789% (3)


In [43]:
df_raw = df_raw[df_raw['ANO_BO'] > 2021]

In [45]:
df_raw = df_raw[~((df_raw['ANO_BO'] == 2026) & (df_raw['MES_BO'] == 7))]

Todos as datas estão **preenchidas**

In [46]:
len(df_raw[df_raw['MES_BO'].isnull() | df_raw['DIA_BO'].isnull() | df_raw['ANO_BO'].isnull()])

0

### 6.2 Construção da Chave Temporal `DATA` (Mensal)

Para a granularidade temporal da análise, é definida uma função que converte a data de cada registro em uma chave no formato `mês-ano` (`MES-ANO`). Essa chave é aplicada linha a linha para gerar a coluna `DATA`, que será usada posteriormente como uma das dimensões de agregação.

In [47]:
def get_date(row):
    return f"{row['MES_BO']}-{row['ANO_BO']}"

In [48]:
df_raw['DATA'] = df_raw.apply(get_date, axis=1)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_19588\1747165987.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_raw['DATA'] = df_raw.apply(get_date, axis=1)


## 7. Tratamento de Registros Duplicados
Será avaliado a presença de registros duplicados e seus impactos

In [49]:
cnt_dedup_records = df_raw.duplicated(subset=['LATITUDE', 'LONGITUDE', 'DATA', 'CIDADE', 'NATUREZA_APURADA', 'NUM_BO']).sum()
pct_dedup_records = cnt_dedup_records / len(df_raw) * 100
print(f"Frequência de Registros Duplicados = {str(pct_dedup_records).replace('.', ',')}% " + f"({cnt_dedup_records:})".replace(",", "."))

Frequência de Registros Duplicados = 0,04607647614519724% (1194)


Cerca de **0,04%** da base são **registros duplicados**. Portanto, como um dado duplicado não agrega informação, logo serão removidos da base.

In [50]:
df_raw = df_raw.drop_duplicates(subset=['LATITUDE', 'LONGITUDE', 'DATA', 'CIDADE', 'NATUREZA_APURADA', 'NUM_BO'], keep='first')

## 8. Tratamento de Coordenadas Geográficas

Antes de qualquer correção, é avaliada a proporção de valores nulos em `LATITUDE` e `LONGITUDE`, individualmente e em conjunto, para dimensionar o problema e orientar a estratégia de imputação.

In [51]:
cnt_lat_null = len(df_raw[df_raw['LATITUDE'].isnull()])
pct_lat_null = cnt_lat_null / len(df_raw) * 100
print(f"Frequência de Registros com Latitude Nula = {str(pct_lat_null).replace('.', ',')}% " + f"({cnt_lat_null:})".replace(",", "."))

Frequência de Registros com Latitude Nula = 11,320695712603516% (293223)


In [52]:
cnt_log_null = len(df_raw[df_raw['LONGITUDE'].isnull()])
pct_log_null = cnt_log_null / len(df_raw) * 100
print(f"Frequência de Registros com Longitude Nula = {str(pct_log_null).replace('.', ',')}% " + f"({cnt_log_null:})".replace(",", "."))

Frequência de Registros com Longitude Nula = 11,320695712603516% (293223)


In [53]:
cnt_lat_log_null = len(df_raw[df_raw['LATITUDE'].isnull() | df_raw['LONGITUDE'].isnull()])
pct_lat_log_null = cnt_lat_log_null / len(df_raw) * 100
print(f"Frequência de Registros com Longitude ou Latitude Nula = {str(pct_lat_log_null).replace('.', ',')}% " + f"({cnt_lat_log_null:})".replace(",", "."))

Frequência de Registros com Longitude ou Latitude Nula = 11,320695712603516% (293223)


### 8.1 Identificando Coordenadas Inválidas

É criada a coluna `COORD_IMPUTADA`, que sinaliza registros cuja coordenada precisa ser tratada/imputada. Um registro é marcado como inválido quando:

- `LATITUDE` ou `LONGITUDE` são nulos; **ou**
- o valor (como string) não começa com `-` (coordenadas no Brasil devem ser negativas); **ou**
- o valor é literalmente o caractere `'-'` (sem dígitos).

In [54]:
df_raw['COORD_IMPUTADA'] = df_raw['LATITUDE'].isnull() | df_raw['LONGITUDE'].isnull() | (~df_raw['LATITUDE'].astype(str).str.match(r'^-')) | (~df_raw['LONGITUDE'].astype(str).str.match(r'^-')) | (df_raw['LATITUDE'] == '-') | (df_raw['LONGITUDE'] == '-')

### 8.2 Funções de Correção de Coordenadas

São definidas três funções auxiliares:

- **`fix_coord_lat` / `fix_coord_long`**: zeram a coordenada de registros marcados como `COORD_IMPUTADA`, preparando-os para receberem o valor definitivo mais adiante;
- **`fix_coord`**: corrige erros de **escala** — coordenadas gravadas sem o ponto decimal (ex: `-4664112.0` em vez de `-46.64112`) — dividindo o valor por 10 repetidamente até que seu módulo fique dentro do limite geográfico válido (90° para latitude, 180° para longitude).

In [55]:
def fix_coord_lat(row):
    return 0 if row['COORD_IMPUTADA'] else row['LATITUDE']

def fix_coord_long(row):
    return 0 if row['COORD_IMPUTADA'] else row['LONGITUDE']

def fix_coord(value, limit=90):
    fixed_value = value
    while abs(fixed_value) > limit:
        fixed_value /= 10
    return fixed_value

Substitui valores **inválidos** por *0*

In [56]:
df_raw['LONGITUDE'] = df_raw.apply(fix_coord_long, axis=1)
df_raw['LATITUDE'] = df_raw.apply(fix_coord_lat, axis=1)

In [57]:
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].astype('Float64')
df_raw['LATITUDE'] = df_raw['LATITUDE'].astype('Float64')

**Corrigi** as coordenadas para um **intervalo válido**

In [58]:
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].map(fix_coord)
df_raw['LATITUDE'] = df_raw['LATITUDE'].map(fix_coord)

In [59]:
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].replace(0, np.nan)
df_raw['LATITUDE'] = df_raw['LATITUDE'].replace(0, np.nan)

### 8.3 Imputação por Centróide da Cidade

Para os registros sem coordenada válida, a estratégia adotada é preencher com a **mediana** de latitude/longitude dos demais registros da mesma cidade (`CIDADE`). A mediana é usada em vez da média por ser mais robusta a eventuais outliers remanescentes na base.

In [60]:
median_point_city = (
    df_raw.dropna(subset=['LATITUDE', 'LONGITUDE'])
    .groupby('CIDADE')[['LATITUDE', 'LONGITUDE']]
    .median()
)

In [61]:
df_raw['LATITUDE'] = df_raw['LATITUDE'].fillna(df_raw['CIDADE'].map(median_point_city['LATITUDE']))
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].fillna(df_raw['CIDADE'].map(median_point_city['LONGITUDE']))

## 9. Junção de Categorias

A coluna `NATUREZA_APURADA` apresenta poucas variações, e nenhuma outra coluna da base permite identificar o tipo específico de delito. Por isso, o trabalho adotará categorias mais genéricas, agrupando as ocorrências em **FURTO** e **ROUBO**.

In [62]:
list_furto = ['FURTO - OUTROS', 'FURTO DE VEÍCULO', 'FURTO DE CARGA']

Padronização da coluna `NATUREZA_APURADA`

In [63]:
df_raw['NATUREZA_APURADA'] = df_raw['NATUREZA_APURADA'].map(lambda x: 'FURTO' if x in list_furto else 'ROUBO')

In [64]:
df_raw['NATUREZA_APURADA'].unique()

array(['FURTO', 'ROUBO'], dtype=object)

## 10. Agregação Final

Com a base limpa e padronizada, o último passo é agregá-la por **LATITUDE, LONGITUDE e DATA(MENSAL)**, gerando as métricas que servirão de base para o cálculo do risco de criminalidade:

- `cnt_total`: contagem **total** de ocorrências por combinação de LATITUDE/LONGITUDE/DATA;
- `cnt_by_type`: contagem de ocorrências **por tipo de crime** (`NATUREZA_APURADA`), uma coluna por categoria, para a mesma combinação de chaves.

In [65]:
cnt_total = (
    df_raw.groupby(['LATITUDE', 'LONGITUDE', 'DATA', 'CIDADE'])
        .size()
        .reset_index(name='CNT_CRIME')
)

In [66]:
cnt_by_type = (
    df_raw.pivot_table(
        index=['LATITUDE', 'LONGITUDE', 'DATA', 'CIDADE'],
        columns='NATUREZA_APURADA',
        aggfunc='size',
        fill_value=0
    ).reset_index()
)

As duas agregações (`cnt_total` e `cnt_by_type`) são unidas em um único dataframe final (`df_group`), contendo a contagem total e a contagem por tipo de crime para cada combinação de localização e data.

In [67]:
df_group = cnt_total.merge(cnt_by_type, on=['LATITUDE', 'LONGITUDE', 'DATA', 'CIDADE'])

## 11. Junção das ocorrências com os indicadores socioeconômicos

In [68]:
gdf_bos = gpd.GeoDataFrame(
    df_group,
    geometry=gpd.points_from_xy(df_group['LONGITUDE'], df_group['LATITUDE']),
    crs='EPSG:4326'
)

In [69]:
gdf_setores_completo = gdf_setores_completo.to_crs('EPSG:4326')

In [70]:
df_bos_enriquecido = gpd.sjoin(
    gdf_bos, 
    gdf_setores_completo, 
    how='left', 
    predicate='within'
)

In [71]:
df_bos_enriquecido = df_bos_enriquecido.drop(columns=['geometry', 'index_right'])

In [72]:
total_bos = len(df_bos_enriquecido)
cruzados_ok = df_bos_enriquecido['CD_SETOR'].notna().sum()
print(f"Total de BOs: {total_bos} | Cruzados com sucesso: {cruzados_ok} ({cruzados_ok/total_bos:.1%})")

Total de BOs: 1739466 | Cruzados com sucesso: 1739458 (100.0%)


Como o número de registros sem correspondência na base socioeconômica foi pequeno, esses casos foram descartados.

In [73]:
df_bos_enriquecido = df_bos_enriquecido[df_bos_enriquecido['CD_SETOR'].notna()]

In [74]:
len(df_bos_enriquecido)

1739458

***Salvar a base agregada***

In [75]:
df_bos_enriquecido.to_csv(data_path + 'database_processed.csv', index=False)